# CryptoGraph — Temporal Graph Network (TGN) on the GPU

Our main model. Every time a coin has a price shock, the TGN gets an event "coin *i* shocked at hour *t*" for each other
coin *j*, and predicts whether *j* shocks within 1, 3, 6, 12 and 24 hours. It keeps a **memory** per coin that is updated
after each batch of events, so it only ever uses the past. We already trained it on a laptop CPU (seed 0: validation
AUPRC 0.153 at 1 hour); this run checks that the GPU gives the same result.

**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Needs `cryptograph_data_v2.zip` in
`MyDrive/CryptoGraph`. About 1–2 hours.

## 1. Install PyTorch Geometric and load the data

In [ ]:
!pip install -q torch_geometric

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CryptoGraph'
!unzip -q -o {DRIVE}/cryptograph_data_v2.zip 'data/processed/v2/1h/panel.pt' 'data/processed/v2/1h/tgn_events.pt' 'data/processed/v2/1h/network_features.pt' -d /content
D = '/content/data/processed/v2/1h'
!ls -lh $D

In [ ]:
import os, copy, time
import torch
import numpy as np, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def focal_loss(logits, target, mask, gamma=2.0, alpha=0.25):
    # focal loss for rare positives (Lin et al. 2017); only entries where mask is True count
    t = target.float()
    bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
    p = torch.sigmoid(logits)
    p_t = p * t + (1 - p) * (1 - t)
    loss = (alpha * t + (1 - alpha) * (1 - t)) * (1 - p_t) ** gamma * bce
    m = mask.float()
    return (loss * m).sum() / m.sum().clamp_min(1.0)

def auprc(y, s):
    return float(average_precision_score(y, s)) if y.sum() > 0 else float('nan')

def auroc(y, s):
    return float(roc_auc_score(y, s)) if 0 < y.sum() < len(y) else float('nan')

SAVE = f'{DRIVE}/results_tgn'
os.makedirs(SAVE, exist_ok=True)

## 2. Events, node features and splits

In [ ]:
panel = torch.load(f'{D}/panel.pt', weights_only=False)
ev = torch.load(f'{D}/tgn_events.pt', weights_only=False)
net = torch.load(f'{D}/network_features.pt', weights_only=False)
data, GIDX, K_HOURS = ev['full'], ev['graph_index'], ev['label_k_hours']
print(f"{data.num_events:,} shock events; message features {ev['msg_names']}; horizons {K_HOURS} h")

# node features at hour t = 6 panel features + 8 network features from the latest daily snapshot at or before t
Xp = panel['X'][:, GIDX].float()
pos = np.searchsorted(net['snapshot_t'].numpy(), np.arange(Xp.shape[0]), side='right') - 1
NF = torch.nan_to_num(net['X'].float())
Xn = torch.zeros(Xp.shape[0], NF.shape[1], NF.shape[2]); Xn[torch.from_numpy(pos >= 0)] = NF[torch.from_numpy(pos[pos >= 0])]
FEATS = torch.cat([Xp, Xn], dim=-1)

# events are sorted by time; train/val/test ranges, and which hours may be scored (embargo, label inside the split)
sid = panel['split_id'].numpy()[data.t.numpy()]
BOUNDS = {s: (int(np.searchsorted(sid, i, 'left')), int(np.searchsorted(sid, i, 'right'))) for i, s in enumerate(['train', 'val', 'test'])}
SCORED = {s: panel['sample_mask'][s].numpy() for s in ['train', 'val', 'test']}
print({s: b[1] - b[0] for s, b in BOUNDS.items()}, 'events (train / val / test)')

## 3. Model

In [ ]:
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import TGNMemory, IdentityMessage, LastAggregator, TimeEncoder, LastNeighborLoader

class GraphAttentionEmbedding(nn.Module):
    # attention over each coin's most recent events, with the time since its last update encoded
    def __init__(self, in_ch, out_ch, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        self.conv = TransformerConv(in_ch, out_ch // 2, heads=2, dropout=0.1, edge_dim=msg_dim + time_enc.out_channels)

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = (last_update[edge_index[0]] - t).to(x.dtype)
        return self.conv(x, edge_index, torch.cat([self.time_enc(rel_t), msg], dim=-1))

class ContagionTGN(nn.Module):
    # event (coin i shocks at hour t, receiver j) -> one logit per horizon k: will j shock within k hours?
    def __init__(self, num_nodes, msg_dim, feat_dim, memory_dim=64, time_dim=16, emb_dim=64, n_out=5, dropout=0.2):
        super().__init__()
        self.memory = TGNMemory(num_nodes, msg_dim, memory_dim, time_dim,
                                message_module=IdentityMessage(msg_dim, memory_dim, time_dim), aggregator_module=LastAggregator())
        self.gnn = GraphAttentionEmbedding(memory_dim, emb_dim, msg_dim, TimeEncoder(time_dim))
        self.head = nn.Sequential(nn.Linear(2 * emb_dim + 2 * feat_dim + msg_dim, emb_dim), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(emb_dim, n_out))

    def embed(self, n_id, edge_index, t_nb, msg_nb):
        z, last_update = self.memory(n_id)
        return self.gnn(z, last_update, edge_index, t_nb, msg_nb)

## 4. Processing the event stream in time order

In [ ]:
T_NP = data.t.numpy()

def batches(start, end, size=200):
    # consecutive slices of ~200 events that never split one hour
    out, a = [], start
    while a < end:
        b = min(a + size, end)
        while b < end and T_NP[b] == T_NP[b - 1]:
            b += 1
        out.append((a, b)); a = b
    return out

class Stream:
    # processes events in time order: predict a batch with memory that only holds EARLIER events, then add the batch
    def __init__(self, model):
        self.model = model
        self.loader = LastNeighborLoader(model.memory.num_nodes, size=10, device=DEV)
        self.assoc = torch.empty(model.memory.num_nodes, dtype=torch.long, device=DEV)
        self.pos_of_eid = torch.zeros(len(T_NP), dtype=torch.long)

    def reset(self):
        self.model.memory.reset_state(); self.loader.reset_state(); self.n = 0

    def run(self, start, end, split, opt=None):
        train = opt is not None
        self.model.train(train)
        logits_all, y_all, m_all, losses = [], [], [], []
        for a, b in batches(start, end):
            src, dst, t, msg = data.src[a:b].to(DEV), data.dst[a:b].to(DEV), data.t[a:b].to(DEV), data.msg[a:b].to(DEV)
            scored = torch.from_numpy(SCORED[split][T_NP[a:b]])
            y, m = data.y[a:b], data.y_mask[a:b] & scored[:, None]
            with torch.set_grad_enabled(train):
                n_id, edge_index, e_id = self.loader(torch.cat([src, dst]).unique())
                self.assoc[n_id] = torch.arange(n_id.size(0), device=DEV)
                p = self.pos_of_eid[e_id.cpu()]
                z = self.model.embed(n_id, edge_index, data.t[p].to(DEV), data.msg[p].to(DEV))
                tt = data.t[a:b]
                x_src, x_dst = FEATS[tt, data.src[a:b]].to(DEV), FEATS[tt, data.dst[a:b]].to(DEV)
                logits = self.model.head(torch.cat([z[self.assoc[src]], z[self.assoc[dst]], x_src, x_dst, msg], dim=-1))
                if m.any():
                    loss = focal_loss(logits, y.to(DEV), m.to(DEV))
                    if train:
                        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(self.model.parameters(), 1.0); opt.step()
                    losses.append(loss.item())
            # only now does the model learn about this batch's events
            self.model.memory.update_state(src, dst, t, msg); self.loader.insert(src, dst)
            self.pos_of_eid[self.n:self.n + (b - a)] = torch.arange(a, b); self.n += b - a
            self.model.memory.detach()
            if not train and m.any():
                logits_all.append(logits.detach().cpu()); y_all.append(y); m_all.append(m)
        if train:
            return np.mean(losses)
        return torch.cat(logits_all), torch.cat(y_all), torch.cat(m_all)

def metrics(split, out):
    logits, y, m = out
    p = torch.sigmoid(logits).numpy()
    res = {}
    for i, k in enumerate(K_HOURS):
        mk = m[:, i].numpy()
        res[f'{split}_auprc_k{k}h'] = auprc(y[:, i].numpy()[mk], p[mk, i])
        res[f'{split}_auroc_k{k}h'] = auroc(y[:, i].numpy()[mk], p[mk, i])
        res[f'{split}_random_k{k}h'] = float(y[:, i].numpy()[mk].mean())
    return res

## 5. Training
Up to 100 epochs; stops after 10 epochs without improvement in validation AUPRC (1 hour) and keeps the best epoch. The test period is scored once at the end.

In [ ]:
EPOCHS, PATIENCE, LR, WD = 100, 10, 5e-4, 1e-4               # same as configs/experiment/tgn_1h_final_v2.yaml

def train_tgn(seed):
    set_seed(seed)
    model = ContagionTGN(len(GIDX), data.msg.shape[1], FEATS.shape[-1], n_out=len(K_HOURS)).to(DEV)
    stream = Stream(model)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    best, best_score, bad, history = None, -np.inf, 0, []
    for epoch in range(EPOCHS):
        t0 = time.time()
        stream.reset()
        train_loss = stream.run(*BOUNDS['train'], 'train', opt)
        val = metrics('val', stream.run(*BOUNDS['val'], 'val'))      # validation continues the event stream
        history.append({'epoch': epoch, 'train_loss': train_loss, **val})
        print(f"epoch {epoch:3d}  train loss {train_loss:.5f}  val AUPRC (1h) {val['val_auprc_k1h']:.4f}  ({time.time() - t0:.0f}s)")
        if val['val_auprc_k1h'] > best_score:
            best, best_score, bad, best_epoch = copy.deepcopy(model.state_dict()), val['val_auprc_k1h'], 0, epoch
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'early stop (best epoch {best_epoch})'); break
    model.load_state_dict(best)
    stream.reset()
    stream.run(*BOUNDS['train'], 'train')                    # replay train to rebuild memory, then score val and test
    result = {'seed': seed, 'best_epoch': best_epoch}
    for split in ['val', 'test']:
        result.update(metrics(split, stream.run(*BOUNDS[split], split)))
    torch.save({'model_state': best, 'result': result, 'history': history}, f'{SAVE}/tgn_1h_seed{seed}.pt')
    return result

## 6. Train (seed 0) and save the model

In [ ]:
result = train_tgn(0)
res = pd.DataFrame([result])
res.to_csv(f'{SAVE}/results.csv', index=False)
display(res.T.round(4))

## 7. Compare with the laptop CPU run (same seed)

In [ ]:
cpu = {'val_auprc_k1h': 0.1528, 'val_auprc_k24h': 0.6538}
for k, v in cpu.items():
    print(f'{k}: GPU {result[k]:.4f} vs CPU {v:.4f}')
!cd $DRIVE && zip -q -r results_tgn.zip results_tgn
print('Saved. Send back: MyDrive/CryptoGraph/results_tgn.zip')